# Configurations and Imports

In [ ]:
import sys
from pathlib import Path

projeto_raiz = Path("~/Documentos/ESPECT_CONV").expanduser().resolve()

if str(projeto_raiz) not in sys.path:
    sys.path.append(str(projeto_raiz))

# ==========================================================
# CONFIGURAÇÕES DE CAMINHO & IMPORTS LOCAIS
# ==========================================================

from pipeline.stage_20_load_and_epoching.stage_20_load_and_epoching import load_and_epoching

from pipeline.stage_30_psd_array_assembly.channels_2D_distribution import CENTRALIZED_ONE_BY_ONE_2D_DISTRIBUTION
from pipeline.stage_30_psd_array_assembly.stage_30_psd_array_assembly import psd_array_assembly
from pipeline.stage_40_obtain_indices.obtain_indices import obtain_indices
from pipeline.stage_50_execute_branch.execute_branch import execute_branch
from pipeline.stage_60_execute_fusion.execute_fusion_majority_voting import execute_fusion_majority_voting
from pipeline.stage_60_execute_fusion.execute_fusion import execute_fusion
from pipeline.stage_70_model_results_comparison.comparison import model_results_comparison

# ======================================================
# CONDITION OPTIONS
# ======================================================
#
# "PRONOUNCED_SPEECH"
#
# "INNER_SPEECH"
#
# "VISUALIZED_CONDITION"
#
# ======================================================

CONDITION = "INNER_SPEECH"

# ==========================================
# SUBJECTS
# ==========================================
# subjects = [
#     f"sub-{i:02d}"
#     for i in range(1, 11)
# ]

subjects = [ "sub-01"]

# ===============================
# TRAINING CONFIGURATIONS
# ===============================

N_FOLDS = 6
VALIDATION_SIZE = 0.20

BATCH_SIZE = 8
LEARNING_RATE = 0.001
RANDOM_STATE = 42
RANDOM_STATE_LOADER = 42





# Stage 20: Load and Epoching

In [ ]:

EPOCH_START_SECONDS = 1.0
EPOCH_END_SECONDS = 3.5

load_and_epoching(
        condition=CONDITION,
        crop_start_time=EPOCH_START_SECONDS,
        crop_end_time=EPOCH_END_SECONDS
)

# Stage 30: PSD Array Assembly

In [ ]:
WINDOW_DURATION = 0.5
STEP_DURATION = 0.25

psd_array_assembly(
        window_duration = WINDOW_DURATION,
        step_duration = STEP_DURATION,
        channels_distribution = CENTRALIZED_ONE_BY_ONE_2D_DISTRIBUTION,
        condition_folder = CONDITION,
        subjects = subjects
)

CENTRALIZED_ONE_BY_ONE_2D_DISTRIBUTION.plot_channels_distribution()

# Stage 40: Obtain Indices

In [ ]:

obtain_indices(
    subjects=subjects,
    N_FOLDS=N_FOLDS,
    VALIDATION_SIZE=VALIDATION_SIZE,
    RANDOM_STATE=RANDOM_STATE,
    condition_folder = CONDITION,
)

# Stage 50: Execute Branch

In [ ]:

# ======================================================
# BRANCH OPTIONS
# ======================================================
#
# space1_space2_frequency_time
#
# space1_space2_time_frequency
#
# time_frequency_space2_space1
#
# time_frequency_space1_space2
#
# ======================================================

branches = [
    "space1_space2_frequency_time", 
    "space1_space2_time_frequency", 
    "time_frequency_space2_space1", 
    "time_frequency_space1_space2" 
]

N_EPOCHS = 2500

for subject in subjects:
    for branch in branches:

        execute_branch(
            branch=branch,
            subjects= subject,
            condition_folder=CONDITION,
            N_FOLDS= N_FOLDS,
            N_EPOCHS=N_EPOCHS,  
            BATCH_SIZE=BATCH_SIZE,  
            LEARNING_RATE=LEARNING_RATE,
            RANDOM_STATE=RANDOM_STATE,
            RANDOM_STATE_LOADER=RANDOM_STATE_LOADER,     
            
        )    

# Stage 60: Execute Fusion

In [ ]:

# ======================================================
# BRANCH_MODEL_STATE OPTIONS
# ======================================================
#
# minimum_loss_model
#
# maximum_accuracy_model
#
# ======================================================

branch_mode_states = ["minimum_loss_model", "maximum_accuracy_model"]

# ======================================================
# FUSION OPTIONS
# ======================================================
#
# mean_fusion
# concatenation_fusion
#
# branch_wise_weighted_sum_fusion
# branch_and_class_wise_weighted_sum_fusion
# 
# branch_wise_scalar_gated_fusion
# branch_and_class_wise_scalar_gated_fusion 
#
# ======================================================

fusions = [
    "mean_fusion",
    "concatenation_fusion",
    "branch_wise_weighted_sum_fusion",
    "branch_and_class_wise_weighted_sum_fusion",
    "branch_wise_scalar_gated_fusion",
    "branch_and_class_wise_scalar_gated_fusion",
]

N_EPOCHS_STAGE_1=1000
N_EPOCHS_STAGE_2=1000
LEARNING_RATE_STAGE_1=0.001
LEARNING_RATE_STAGE_2=0.001

# Majority voting
for subject in subjects:
    for branch_mode_state in branch_mode_states:

        execute_fusion_majority_voting(
            fusion= "majority_voting_fusion",
            branch_model_state = branch_mode_state,
            subjects=subject,
            condition_folder = CONDITION, 
            N_FOLDS=N_FOLDS,
            BATCH_SIZE=BATCH_SIZE,
            RANDOM_STATE=RANDOM_STATE,       
        )

# Fusions
for subject in subjects:    
    for fusion in fusions:
        for branch_mode_state in branch_mode_states:

            execute_fusion(
                fusion= fusion,
                branch_model_state = branch_mode_state,
                subjects=subject,
                condition_folder = CONDITION,
                N_FOLDS=N_FOLDS,
                BATCH_SIZE=BATCH_SIZE,
                N_EPOCHS_STAGE_1 = N_EPOCHS_STAGE_1,
                N_EPOCHS_STAGE_2 = N_EPOCHS_STAGE_2,
                LEARNING_RATE_STAGE_1 = LEARNING_RATE_STAGE_1,
                LEARNING_RATE_STAGE_2 = LEARNING_RATE_STAGE_2,
                RANDOM_STATE=RANDOM_STATE,
                RANDOM_STATE_LOADER=RANDOM_STATE_LOADER,
            )

# Stage 70: Model Results Comparison

In [ ]:

# ======================================================
# BRANCH OPTIONS
# ======================================================
#
# space1_space2_frequency_time
#
# space1_space2_time_frequency
#
# time_frequency_space2_space1
#
# time_frequency_space1_space2
#
# ======================================================

# ======================================================
# BRANCH_MODEL_STATE OPTIONS
# ======================================================
#
# minimum_loss_model
#
# maximum_accuracy_model
#
# ======================================================

# ======================================================
# FUSION OPTIONS
# ======================================================
#
# mean_fusion
# concatenation_fusion
#
# branch_wise_weighted_sum_fusion
# branch_and_class_wise_weighted_sum_fusion
# 
# branch_wise_scalar_gated_fusion
# branch_and_class_wise_scalar_gated_fusion 
#
# ======================================================


model_results_comparison(     
    subjects= subjects,
    condition_folder = CONDITION,
    N_FOLDS=N_FOLDS,
    model_states = ["minimum_loss_model", "maximum_accuracy_model"],
    branches= [
        "space1_space2_frequency_time",
        "space1_space2_time_frequency",
        "time_frequency_space2_space1",
        "time_frequency_space1_space2"
        ],
    majority_voting_fusions = [
        "majority_voting_fusion"
    ],
    fusions = [
        "mean_fusion",  
        "concatenation_fusion",
        "branch_wise_weighted_sum_fusion", 
        "branch_and_class_wise_weighted_sum_fusion",
        "branch_wise_scalar_gated_fusion",
        "branch_and_class_wise_scalar_gated_fusion"    
    ],

)